In [1]:
import pandas as pd

square_simple_test = pd.read_csv("C:/Studia/6 sem/MIOWAD/mio1/regression/square-simple-test.csv")
square_simple_training = pd.read_csv("C:/Studia/6 sem/MIOWAD/mio1/regression/square-simple-training.csv")
steps_large_test = pd.read_csv("C:/Studia/6 sem/MIOWAD/mio1/regression/steps-large-test.csv")
steps_large_training = pd.read_csv("C:/Studia/6 sem/MIOWAD/mio1/regression/steps-large-training.csv")

In [2]:
square_simple_training.head()

,Unnamed: 0,x,y
0,1,-0.171543,-127.351580
1,2,0.025201,-129.942844
2,3,-1.368991,38.672367
3,4,1.907390,197.432191
4,5,0.011129,-129.988852


In [3]:
steps_large_training.head()

,Unnamed: 0,x,y
0,1,-1.481354,-80
1,2,1.033264,80
2,3,-0.076403,0
3,4,-1.419785,-80
4,5,-0.108398,0


In [4]:
# square-simple
# x_train = square_simple_training[['x']].values
# y_train = square_simple_training[['y']].values

# x_test = square_simple_test[['x']].values
# y_test = square_simple_test[['y']].values

# steps-large
x_train = steps_large_training[['x']].values
y_train = steps_large_training[['y']].values

x_test = steps_large_test[['x']].values
y_test = steps_large_test[['y']].values

# architektura sieci

In [5]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # bo 2 pakiety maja taka sama biblioteke
import csv
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

In [6]:
class MLP:
    
    def __init__(self, layer_sizes: list, activation: str = 'sigmoid'):
        self.layer_sizes = layer_sizes  # np. [1, 5, 1] - 1 wejście, 5 neuronow ukrytych, 1 wyjście
        self.activation_name = activation
        self.biases = []
        self.weights = []
    
    def _activation(self, x):
        if self.activation_name == 'sigmoid':
            return 1 / (1 + np.exp(-x))
        elif self.activation_name == 'tanh':
            return np.tanh(x)
        elif self.activation_name == 'relu':
            return np.maximum(0, x)
        else:
            raise ValueError('nieznana funkcja aktywacji')
            
    def set_activation(self, activation: str):
        self.activation_name = activation

    def set_weights(self, layer_index: int, weights: np.ndarray):
        self.weights[layer_index] = weights

    def set_bias(self, layer_index: int, bias: np.ndarray):
        self.biases[layer_index] = bias.reshape(1, -1)

    def forward(self, x: np.ndarray) -> np.ndarray:
        x2 = x
        for i in range(len(self.weights) - 1):  # warstwy ukryte
            z = x2 @ self.weights[i] + self.biases[i]
            x2 = self._activation(z)
        z2 = x2 @ self.weights[-1] + self.biases[-1] #warstwa wyjsciowa - fun aktywacji jest liniowa
        return z2
          
    def load_weights(self, pytorch_model): 
        linear_layers = [m for m in pytorch_model.modules() if isinstance(m, nn.Linear)]  # pobieramy warstwy z pytorcha
        self.weights = [] 
        self.biases = []

        for layer in linear_layers:  # z kolejnych warstw wyciągamy wagi i biasy
            self.weights.append(layer.weight.data.cpu().numpy().T)
            self.biases.append(layer.bias.data.cpu().numpy().reshape(1, -1))

In [7]:
# architektura:
# 1 - jedna warstwa ukryta, 5 neuronów,
# 2- jedna warstwa ukryta, 10 neuronów,
# 3 - dwie warstwy ukryte, po 5 neuronów każda.
architecture = "3"
normalization = True
epochs = 30000

if architecture == '1':
    layers = [1, 5, 1]
elif architecture == '2':
    layers = [1, 10, 1]
elif architecture == '3':
    layers = [1, 5, 5, 1]


# model pytorch

In [8]:
def build_model(arch):
    if arch == '1':
        return nn.Sequential(nn.Linear(1,5), nn.Sigmoid(), nn.Linear(5,1))
    elif arch == '2':
        return nn.Sequential(nn.Linear(1,10), nn.Sigmoid(), nn.Linear(10,1))
    elif arch == '3':
        return nn.Sequential(nn.Linear(1,5), nn.Sigmoid(), nn.Linear(5,5), nn.Sigmoid(), nn.Linear(5,1))

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model = build_model(architecture).to(device)


In [9]:
x_tr = torch.tensor(x_train, dtype=torch.float32).reshape(-1,1).to(device)
y_tr = torch.tensor(y_train, dtype=torch.float32).reshape(-1,1).to(device)

# normalizacja
x_min, x_max = x_tr.min(), x_tr.max()
y_min, y_max = y_tr.min(), y_tr.max()

if normalization:
    x_norm = (x_tr - x_min) / (x_max - x_min)
    y_norm = (y_tr - y_min) / (y_max - y_min)
else:
    x_norm, y_norm = x_tr, y_tr

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

#trenowanie
for i in range(epochs):
    y_pred = model(x_norm)
    loss = criterion(y_pred, y_norm)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

#do nieznormalizowanych
with torch.no_grad():
    y_pred_train = model(x_norm)
    if normalization:
        y_pred_train_denorm = y_pred_train * (y_max - y_min) + y_min
    else:
        y_pred_train_denorm = y_pred_train
    mse_train = criterion(y_pred_train_denorm, y_tr).item()

print(f'MSE na danych treningowych, nieznormalizowanych: {mse_train:.4f}')

MSE na danych treningowych, nieznormalizowanych: 7.8023


# sprawdzamy model pytorch na danych testowych

In [10]:
x_te = torch.tensor(x_test, dtype=torch.float32).reshape(-1,1).to(device)
y_te = torch.tensor(y_test, dtype=torch.float32).reshape(-1,1).to(device)

if normalization:
    x_te_norm = (x_te - x_min) / (x_max - x_min)
else:
    x_te_norm = x_te

with torch.no_grad():
    y_pred_te = model(x_te_norm)
    if normalization:
        y_pred_te_denorm = y_pred_te * (y_max - y_min) + y_min
    else:
        y_pred_te_denorm = y_pred_te

mse_pytorch_test = criterion(y_pred_te_denorm, y_te).item()
print(f'MSE na danych testowych, nieznormalizowanych: {mse_pytorch_test:.4f}')

MSE na danych testowych, nieznormalizowanych: 6.9859


# wyciąganie wag

In [11]:
new_mlp = MLP(layer_sizes=layers, activation='sigmoid')
new_mlp.load_weights(model)

# forward na zaimplementowanym MLP i porównanie

In [12]:
x_te_impl = np.array(x_test).reshape(-1,1)
y_te_impl = np.array(y_test).reshape(-1,1)

# przetwarzamy na danych znormalizowanych
if normalization:
    x_te_norm_impl = (x_te_impl - x_min.item()) / (x_max.item() - x_min.item()) 
else:
    x_te_norm_impl = x_te_impl

y_pred_impl = new_mlp.forward(x_te_norm_impl)

# powrót do danych nieznormalizowanych
if normalization:
    y_pred_impl_denorm = y_pred_impl * (y_max.item() - y_min.item()) + y_min.item()
else:
    y_pred_impl_denorm = y_pred_impl

mse_impl_test = np.mean((y_te_impl - y_pred_impl_denorm)**2)
print(f'MSE z pytorch: {mse_pytorch_test}')
print(f'MSE z implementacji: {mse_impl_test:.4f}')

MSE z pytorch: 6.985885143280029
MSE z implementacji: 6.9858
